# Notebook 11b — Ablación del NER: ¿qué está leyendo realmente?

**Proyecto BME513 · Universidad de Valparaíso** — Sebastián Inostroza Hurtado

---

## Por qué hago este experimento

El NER da **F1 de span = 0,9991** sobre el corpus. Es una cifra alta, y las cifras altas hay que interrogarlas.

Al auditar de dónde salen las etiquetas apareció esto:

```
La columna Recommendations aparece textual dentro del Full_Report : 99,8 %
Los informes traen el encabezado "RECOMENDACIONES:"               : 99,8 %
Un regex de UNA LÍNEA reproduce la columna exactamente            : 99,8 %
```

O sea: la etiqueta del NER es, en la práctica, **"lo que viene después de `RECOMENDACIONES:`"**. Y el NER (0,9991) apenas le gana a ese regex trivial (0,998).

**Es el mismo patrón que llevó a descartar el verificador de BI-RADS.** Allá, la ablación mostró que el modelo leía el número declarado en lugar de deducirlo (0,939 → 0,544), y se retiró del pipeline.

Corresponde aplicar aquí **el mismo estándar**. Si no, estaría exigiéndole a un modelo una prueba que al otro le perdono.

## La pregunta

¿El NER aprendió a buscar el **encabezado**, o aprendió a reconocer el **contenido** de una recomendación clínica?

## El experimento

**Ablación**: se toma el modelo ya entrenado, se enmascara el encabezado `RECOMENDACIONES:` en el conjunto de prueba, y se reevalúa. El modelo **no se reentrena**: es el mismo, leyendo una entrada mutilada.

| Condición | Entrada de prueba |
|---|---|
| **A · control** | El informe tal cual. Debe reproducir el 0,9991 |
| **B · sin encabezado** | Igual, con `RECOMENDACIONES:` reemplazado por espacios |

## Hipótesis y qué haría con cada resultado

| | Resultado | Lectura | Consecuencia |
|---|---|---|---|
| **H1** | F1 se mantiene alto (caída < 0,05) | Aprendió el **contenido**: los verbos, el contexto clínico | El NER queda **justificado con 4 357 casos**, no con los 3 informes chilenos. Su lugar en el sistema se blinda con el mismo estándar que descartó al verificador |
| **H2** | Caída moderada (0,05 a 0,20) | Usa el encabezado como ayuda, pero no depende de él | Aporta, y hay que reportar la dependencia parcial |
| **H3** | Se derrumba (caída > 0,20) | Aprendió el **encabezado** | Su valor descansa solo en 3 informes chilenos. Sería el mismo hallazgo que en el BI-RADS, y habría que decirlo |

## Señal que existe además del encabezado

Medido sobre el corpus, hay una segunda pista que el modelo **podría** haber aprendido:

```
Recomendaciones que empiezan con un verbo gatillo : 98,9 %
  "- se sugiere ...", "- sugerimos ...", "- se recomienda ..."
```

Si el NER aprendió eso, sobrevive a la ablación. Si solo aprendió el encabezado, no.

> **Este notebook es independiente.** No reentrena nada ni modifica ningún archivo. Solo carga el modelo ya guardado y lo evalúa dos veces.


---
## Paso 1 — Configuración

Los valores replican el nb 11 para reconstruir **exactamente** el mismo conjunto de prueba.

In [ ]:
import os, re, json, random, warnings, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

warnings.filterwarnings("ignore")

# ----- Idéntico al nb 11 -----
MODEL_DIR   = "../models/ner_recomendacion_final"
DATA_PATH   = "../data/processed/reports_cleaned.csv"
MAX_LEN     = 384
SEED        = 42
label_list  = ["O", "B-REC", "I-REC"]

# Referencia a reproducir
F1_NB11 = 0.9991

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Rutas alternativas por si la estructura difiere
for cand in [MODEL_DIR, "models/ner_recomendacion_final", "../../models/ner_recomendacion_final"]:
    if Path(cand).exists():
        MODEL_DIR = cand; break
for cand in [DATA_PATH, "data/processed/reports_cleaned.csv", "data/reports_cleaned.csv"]:
    if Path(cand).exists():
        DATA_PATH = cand; break

assert Path(MODEL_DIR).exists(), f"No encuentro el modelo en {MODEL_DIR}"
assert Path(DATA_PATH).exists(), f"No encuentro el CSV en {DATA_PATH}"
print(f"Modelo: {MODEL_DIR}")
print(f"Datos : {DATA_PATH}")

---
## Paso 2 — Reconstruyo el conjunto de prueba del nb 11

Mismo etiquetado BIO, misma deduplicación por firma, mismo split con la misma semilla. Es indispensable: si el test fuera otro, el 0,9991 no sería comparable.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Corpus: {len(df)} informes")

# CRITICO: el nb 11 entreno con las columnas _clean (minusculas, sin tildes,
# una sola linea). El modelo es uncased. Pasarle Full_Report crudo produce
# predicciones basura: F1 ~ 0.002. Deben ser las mismas columnas.
assert "Full_Report_clean" in df.columns and "Recommendations_clean" in df.columns, \
    "Faltan las columnas _clean: el nb 11 entreno con ellas"

def limpiar_rec(rec):
    """COPIADA TEXTUAL del nb 11. No reescribir.

    Quita el guion inicial ("- se sugiere..." -> "se sugiere..."). Ese guion
    es la razón por la que una version anterior de este notebook daba F1 = 0:
    el span de verdad empezaba en "-" y el del modelo en "se", corrido un
    token, y seqeval exige coincidencia exacta de inicio y fin.
    """
    return str(rec).strip().lstrip("-*\u2022 ").strip()

def etiquetar_bio(full_report, recomendacion):
    """Idéntica al nb 11. Etiquetas a nivel de PALABRA."""
    full = str(full_report); rec = limpiar_rec(recomendacion)
    tokens = full.split(); etiquetas = ["O"] * len(tokens)
    rec_tokens = rec.split()
    if not rec_tokens: return tokens, etiquetas
    n, m = len(tokens), len(rec_tokens)
    for i in range(n - m + 1):
        if all(tokens[i+j].strip(".,;:") == rec_tokens[j].strip(".,;:") for j in range(m)):
            etiquetas[i] = "B-REC"
            for j in range(1, m): etiquetas[i+j] = "I-REC"
            break
    return tokens, etiquetas

def firma(tokens):
    """Idéntica al nb 11. Aquí la etiqueta es una POSICIÓN, no un número,
    así que neutralizar dígitos es seguro (a diferencia del nb 04c)."""
    t = " ".join(tokens).lower()
    t = re.sub(r"\d+", "#", t)
    t = re.sub(r"[^a-zñáéíóú# ]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

datos = []
for _, row in df.iterrows():
    toks, labs = etiquetar_bio(row["Full_Report_clean"], row["Recommendations_clean"])
    if "B-REC" in labs:
        datos.append({"tokens": toks, "labels": labs})
print(f"Con span localizable: {len(datos)}")

# Deduplicación por firma (igual que el nb 11)
visto, dedup = set(), []
for d in datos:
    f = firma(d["tokens"])
    if f not in visto:
        visto.add(f); dedup.append(d)
print(f"Tras deduplicar: {len(dedup)}  ({len(datos)-len(dedup)} eliminados, {(len(datos)-len(dedup))/len(datos)*100:.1f}%)")

# Split 70/15/15 con la misma semilla
from sklearn.model_selection import train_test_split
tr, tmp = train_test_split(dedup, test_size=0.30, random_state=SEED, shuffle=True)
va, te  = train_test_split(tmp,  test_size=0.50, random_state=SEED, shuffle=True)
print(f"\nSplit -> train {len(tr)} · val {len(va)} · TEST {len(te)}")

---
## ✋ Checkpoint 1 — ¿Reconstruí bien el test? (no evalúa nada)

Antes de gastar cómputo, verifico que el conjunto de prueba tenga el tamaño esperado y que el encabezado esté presente en él. Si el encabezado no estuviera, la ablación no tendría nada que enmascarar.

In [ ]:
PAT_ENCABEZADO = re.compile(r"RECOMENDACION(?:ES)?\s*:?", re.IGNORECASE)

con_enc = sum(1 for d in te if PAT_ENCABEZADO.search(" ".join(d["tokens"])))
print("="*62)
print("CHECKPOINT 1 — sanidad del conjunto de prueba")
print("="*62)
print(f"  Informes en test                    : {len(te)}")
print(f"  Con encabezado RECOMENDACIONES      : {con_enc}  ({con_enc/len(te)*100:.1f}%)")

verbos = re.compile(r"\b(se sugiere|sugerimos|se recomienda|recomendamos|amerita|procede|se indica)\b", re.I)
con_verbo = sum(1 for d in te
                if verbos.search(" ".join(t for t,l in zip(d["tokens"], d["labels"]) if l != "O")))
print(f"  Con verbo gatillo en el span        : {con_verbo}  ({con_verbo/len(te)*100:.1f}%)")
print()
print("  El encabezado es la señal que voy a tapar.")
print("  El verbo es la señal que quedaría disponible si el modelo la aprendió.")
assert con_enc / len(te) > 0.9, "Poco encabezado en el test: la ablación no tendría sentido"
print("\n  OK: hay encabezado que enmascarar.")

# Ejemplo visual
ej = next(d for d in te if PAT_ENCABEZADO.search(" ".join(d["tokens"])))
txt = " ".join(ej["tokens"])
i = PAT_ENCABEZADO.search(txt).start()
print("\n" + "="*62)
print("EJEMPLO — qué verá el modelo en cada condición")
print("="*62)
print(f"  A (control)      : ...{txt[max(0,i-60):i+90]}...")
print(f"  B (sin encabezado): ...{PAT_ENCABEZADO.sub(' ', txt[max(0,i-60):i+90])}...")

---
## Paso 3 — La ablación: enmascarar el encabezado

Se reemplaza `RECOMENDACIONES:` por espacios. **Las etiquetas no se tocan**: el span sigue siendo el mismo, en la misma posición. Lo único que desaparece es la pista.

Importante: se enmascara **solo el encabezado**, no el contenido. El modelo sigue teniendo todo el texto de la recomendación disponible. Si aprendió a reconocerlo, lo va a encontrar.

In [ ]:
def enmascarar_encabezado(tokens, labels):
    """Quita el token del encabezado y su etiqueta O correspondiente.

    Se elimina el token completo en lugar de vaciarlo, para no dejar
    un token vacío que descoloque el alineamiento palabra-subtoken.
    """
    nuevos_t, nuevos_l = [], []
    for t, l in zip(tokens, labels):
        if PAT_ENCABEZADO.fullmatch(t.strip()) or PAT_ENCABEZADO.fullmatch(t.strip(":-. ")):
            continue                      # se descarta el encabezado
        nuevos_t.append(t); nuevos_l.append(l)
    return nuevos_t, nuevos_l

te_ablado = []
n_tocados = 0
for d in te:
    t2, l2 = enmascarar_encabezado(d["tokens"], d["labels"])
    if len(t2) != len(d["tokens"]): n_tocados += 1
    te_ablado.append({"tokens": t2, "labels": l2})

print(f"Informes de test modificados: {n_tocados}/{len(te)}  ({n_tocados/len(te)*100:.1f}%)")

# Verificación crítica: el span debe sobrevivir intacto
spans_a = [sum(1 for l in d["labels"] if l != "O") for d in te]
spans_b = [sum(1 for l in d["labels"] if l != "O") for d in te_ablado]
iguales = sum(1 for a, b in zip(spans_a, spans_b) if a == b)
print(f"Spans con el mismo largo tras enmascarar: {iguales}/{len(te)}")
assert iguales == len(te), "La ablación alteró algún span: revisar"
print("  OK: los spans quedaron intactos. Solo desapareció el encabezado.")

resta = sum(1 for d in te_ablado if PAT_ENCABEZADO.search(" ".join(d["tokens"])))
print(f"Informes que TODAVÍA tienen encabezado: {resta}  (debe ser 0 o casi)")

---
## Paso 4 — Cargo el modelo entrenado

**No se reentrena.** Es el mismo modelo del nb 11, tal como quedó guardado.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
modelo    = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).to(DEVICE)
modelo.eval()
print(f"Modelo cargado: {sum(p.numel() for p in modelo.parameters())/1e6:.1f} M parámetros")
print(f"Etiquetas: {modelo.config.id2label}")

# --------------------------------------------------------------------
# MAPEO DE ETIQUETAS. Ojo con esto.
#
# El nb 11 creó el modelo con:
#     AutoModelForTokenClassification.from_pretrained(MODELO, num_labels=3)
#
# Sin pasar id2label. Por eso HuggingFace le asignó las etiquetas por
# defecto: {0:'LABEL_0', 1:'LABEL_1', 2:'LABEL_2'}. El nb 11 nunca lo
# notó porque su compute_metrics usaba un id2label propio del notebook.
#
# Conclusión: NO se puede confiar en modelo.config.id2label aquí. El
# orden verdadero es el de label_list, que es el que se usó para
# construir label2id en el entrenamiento:
#     label2id = {l: i for i, l in enumerate(label_list)}
#     -> O = 0, B-REC = 1, I-REC = 2
# --------------------------------------------------------------------
cfg = {int(k): v for k, v in modelo.config.id2label.items()}
print(f"\nid2label guardado en el modelo : {cfg}")

if set(cfg.values()) == set(label_list):
    ID2LABEL = cfg
    print("  El modelo trae las etiquetas reales. Uso las suyas.")
else:
    ID2LABEL = {i: l for i, l in enumerate(label_list)}
    print(f"  El modelo trae etiquetas genéricas. Uso el orden del entrenamiento: {ID2LABEL}")

assert set(ID2LABEL.values()) == set(label_list), "El mapeo no cubre las 3 etiquetas"
assert len(ID2LABEL) == modelo.config.num_labels, "El mapeo no calza con num_labels"
print(f"  Mapeo en uso: {ID2LABEL}")

---
## ✋ Checkpoint 2 — Estimación de tiempo (no evalúa)

Solo inferencia, sin entrenamiento. Debería ser rápido.

In [ ]:
n_total = len(te) * 2   # dos condiciones
seg_por_informe = 0.02  # estimación conservadora en MPS para inferencia
print("="*62)
print("CHECKPOINT 2 — cuánto va a tardar")
print("="*62)
print(f"  Informes a evaluar : {len(te)} x 2 condiciones = {n_total}")
print(f"  Estimación         : ~{n_total*seg_por_informe/60:.1f} minutos")
print("\n  Es solo inferencia: no se entrena nada.")

---
## Paso 5 — La función de predicción

Predice palabra por palabra, tomando el **primer subtoken** de cada palabra. Es exactamente el criterio que usó el nb 11 al entrenar (los subtokens de continuación llevan `-100` y se ignoran).

In [ ]:
@torch.no_grad()
def predecir(tokens):
    """Devuelve las etiquetas predichas, una por palabra."""
    enc = tokenizer(tokens, is_split_into_words=True, truncation=True,
                    max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    logits = modelo(**enc).logits[0]
    pred_sub = logits.argmax(-1).cpu().numpy()
    word_ids = enc.word_ids(0)

    out, visto = [], set()
    for sub_i, w_i in enumerate(word_ids):
        if w_i is None or w_i in visto: continue
        visto.add(w_i)
        out.append(ID2LABEL[int(pred_sub[sub_i])])
    # Si truncó, completar con O
    while len(out) < len(tokens): out.append("O")
    return out[:len(tokens)]

---
## ✋ Checkpoint 3 — Prueba de humo (5 segundos, no evalúa todo)

Antes de gastar 10 minutos, verifico sobre **5 informes** que el modelo predice bien. Si esto falla, el experimento completo también va a fallar, y aquí se ve por qué de inmediato.

Esta celda existe porque las dos primeras versiones de este notebook dieron F1 cercano a cero por dos bugs distintos: el texto equivocado (crudo en vez de `_clean`) y el mapeo de etiquetas equivocado (`LABEL_0` en vez de `O`). Ambos se habrían detectado aquí en segundos.

In [ ]:
from seqeval.metrics import f1_score as _f1

print("="*66)
print("PRUEBA DE HUMO — 5 informes")
print("="*66)

muestra = te[:5]
yt = [d["labels"] for d in muestra]
yp = [predecir(d["tokens"]) for d in muestra]
f1_humo = _f1(yt, yp)

print(f"  F1 sobre 5 informes: {f1_humo:.4f}")
print()

if f1_humo < 0.5:
    print("  FALLA. No sigas: el experimento completo va a dar lo mismo.")
    print()
    d = muestra[0]
    txt = " ".join(d["tokens"])
    print("  1) TEXTO QUE RECIBE EL MODELO:")
    print(f"     {txt[:110]}...")
    print(f"     ¿minúsculas? {txt == txt.lower()}   ¿sin tildes? {not any(c in txt for c in 'áéíóúñ')}")
    print("     -> el modelo es uncased. Deben ser las columnas _clean.")
    print()
    print("  2) ETIQUETAS:")
    print(f"     Verdad   : {[l for l in d['labels'] if l != 'O'][:6]}")
    print(f"     Predicho : {[l for l in yp[0] if l != 'O'][:6]}")
    print(f"     Mapeo    : {ID2LABEL}")
    print("     -> si el predicho dice LABEL_0/LABEL_1, el mapeo está mal.")
    print("     -> si el predicho está vacío, el modelo no reconoce el texto.")
    print()
    print("  3) MODELO:")
    print(f"     Ruta: {MODEL_DIR}")
    import collections
    print(f"     Distribución predicha: {dict(collections.Counter(yp[0]))}")
    raise RuntimeError("Prueba de humo fallida. Revisa lo de arriba antes de seguir.")

print("  OK. El modelo predice bien. Sigo con el experimento completo.")

---
## Paso 6 — Evaluación completa

Se predice palabra por palabra y se compara con seqeval a nivel de **span**, igual que el nb 11. Un span cuenta como acierto solo si coincide completo: mismo inicio y mismo fin.

In [ ]:
def evaluar(conjunto, nombre):
    y_true, y_pred = [], []
    t0 = time.time()
    for i, d in enumerate(conjunto):
        y_true.append(d["labels"])
        y_pred.append(predecir(d["tokens"]))
        if (i+1) % 100 == 0:
            print(f"    {i+1}/{len(conjunto)}...", end="\r")
    dur = time.time() - t0
    r = {
        "f1":        f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall":    recall_score(y_true, y_pred),
        "minutos":   dur/60,
    }
    print(f"  {nombre}: F1 = {r['f1']:.4f} · P = {r['precision']:.4f} · R = {r['recall']:.4f}  ({dur/60:.1f} min)")
    return r, y_true, y_pred

print("="*62)
print("CONDICIÓN A — control (informe tal cual)")
print("="*62)
res_A, yt_A, yp_A = evaluar(te, "A")

print()
print("="*62)
print("CONDICIÓN B — ablación (sin el encabezado RECOMENDACIONES)")
print("="*62)
res_B, yt_B, yp_B = evaluar(te_ablado, "B")

---
## Paso 7 — El veredicto

In [ ]:
delta = res_B["f1"] - res_A["f1"]

print("="*66)
print("RESULTADO DE LA ABLACIÓN DEL NER")
print("="*66)
print(f"  nb 11 (referencia)          : F1 = {F1_NB11:.4f}")
print(f"  A · control                 : F1 = {res_A['f1']:.4f}")
print(f"  B · sin encabezado          : F1 = {res_B['f1']:.4f}")
print(f"  Caída                       : {delta:+.4f}  ({delta/max(res_A['f1'],1e-9)*100:+.1f} %)")

# ---------------------------------------------------------------
# PUERTA DE SANIDAD: si el control no reproduce el nb 11, el
# experimento NO es interpretable y el veredicto NO se emite.
# ---------------------------------------------------------------
CONTROL_OK = abs(res_A["f1"] - F1_NB11) <= 0.02

print()
print("="*66)
if not CONTROL_OK:
    veredicto = "EXPERIMENTO INVÁLIDO — el control no reproduce el nb 11"
    print("EXPERIMENTO INVÁLIDO. NO LEAS LA ABLACIÓN.")
    print("="*66)
    print(f"  La condición A da {res_A['f1']:.4f} y debería dar {F1_NB11:.4f}.")
    print("  Si el control falla, la caída no significa nada: ambas condiciones")
    print("  pueden estar fallando por el mismo bug y la diferencia daría cero.")
    print()
    print("  QUÉ REVISAR, en orden de probabilidad:")
    print("   1. ¿Las columnas son Full_Report_clean y Recommendations_clean?")
    print("      El modelo es uncased y se entrenó con texto en minúsculas y sin")
    print("      tildes. Pasarle el texto crudo da F1 ~ 0.002.")
    print("   2. ¿ID2LABEL coincide con el orden del entrenamiento?")
    print("   3. ¿MODEL_DIR apunta al modelo afinado y no al base?")
    print()
    print("  Corre la celda de diagnóstico que viene a continuación.")
else:
    print("LECTURA")
    print("="*66)
    print(f"  OK: la condición A reproduce el nb 11. El test está bien reconstruido.")
    print()
    if abs(delta) < 0.05:
        veredicto = "H1 — APRENDIÓ EL CONTENIDO"
        print(f"  {veredicto}")
        print("\n  Quitar el encabezado casi no lo afecta. El modelo no estaba buscando")
        print("  'RECOMENDACIONES:': aprendió a reconocer qué ES una recomendación.")
        print("\n  CONSECUENCIA: el NER queda justificado con los 4 357 informes del corpus,")
        print("  no solo con los 3 chilenos. Su lugar en el sistema se sostiene con el mismo")
        print("  estándar que se usó para descartar el verificador de BI-RADS.")
        print("\n  El contraste:")
        print("     Verificador BI-RADS, sin el número : 0,939 -> 0,544   (se derrumba)")
        print(f"     NER, sin el encabezado             : {res_A['f1']:.3f} -> {res_B['f1']:.3f}   (aguanta)")
    elif abs(delta) <= 0.20:
        veredicto = "H2 — DEPENDENCIA PARCIAL"
        print(f"  {veredicto}")
        print("\n  El encabezado ayuda, pero el modelo no depende de él por completo.")
        print("  Hay que reportar la dependencia parcial y matizar el argumento de")
        print("  generalización, que sigue en pie pero es más débil de lo que parecía.")
    else:
        veredicto = "H3 — APRENDIÓ EL ENCABEZADO"
        print(f"  {veredicto}")
        print("\n  El modelo se apoyaba en 'RECOMENDACIONES:'. Sin ese ancla, se derrumba.")
        print("  Es el MISMO hallazgo que en el verificador de BI-RADS, y corresponde")
        print("  reportarlo con la misma honestidad.")
    print(f"\n  VEREDICTO: {veredicto}")

if CONTROL_OK:
    print()
    print("="*66)
    print("REPORTE POR SPAN — CONDICIÓN B")
    print("="*66)
    print(classification_report(yt_B, yp_B, digits=4))

---
## Diagnóstico (correr solo si el control falla)

Muestra un ejemplo concreto: qué texto recibe el modelo, qué predice y qué debería predecir. Un F1 cercano a cero casi siempre es un desajuste entre el texto de entrenamiento y el de evaluación, no un hallazgo.

In [ ]:
if not CONTROL_OK:
    d = te[0]
    txt = " ".join(d["tokens"])
    pred = predecir(d["tokens"])

    print("="*66); print("1. ¿QUÉ TEXTO RECIBE EL MODELO?"); print("="*66)
    print(f"  {txt[:180]}...")
    print()
    print("  ¿Está en minúsculas?", txt == txt.lower())
    print("  ¿Tiene tildes?      ", any(c in txt for c in "áéíóúñÁÉÍÓÚ"))
    print("  -> el modelo es UNCASED y se entrenó con texto en minúsculas SIN tildes.")
    print("     Si arriba ves MAYÚSCULAS o tildes, estás usando la columna equivocada.")

    print()
    print("="*66); print("2. ETIQUETAS: VERDAD vs PREDICHO (solo lo no-O)"); print("="*66)
    verdad = [(t,l) for t,l in zip(d["tokens"], d["labels"]) if l != "O"]
    predic = [(t,l) for t,l in zip(d["tokens"], pred)        if l != "O"]
    print(f"  Verdad   ({len(verdad):3d} tokens): {' '.join(t for t,_ in verdad[:12])}...")
    print(f"  Predicho ({len(predic):3d} tokens): {' '.join(t for t,_ in predic[:12]) if predic else '(NADA: predijo todo O)'}")

    print()
    print("="*66); print("3. MAPEO DE ETIQUETAS"); print("="*66)
    print(f"  Del modelo : {ID2LABEL}")
    print(f"  Esperado   : {dict(enumerate(label_list))}")
    print("  -> si no coinciden, el decodificado está invirtiendo las clases.")

    print()
    print("="*66); print("4. MODELO CARGADO"); print("="*66)
    print(f"  Ruta: {MODEL_DIR}")
    import collections
    c = collections.Counter(pred)
    print(f"  Distribución de lo predicho: {dict(c)}")
    print("  -> si predice casi todo 'O', o el texto no coincide con el de")
    print("     entrenamiento, o MODEL_DIR apunta al modelo base sin afinar.")
else:
    print("El control reproduce el nb 11. No hace falta diagnosticar.")

---
## Paso 8 — Guardo el resultado

In [ ]:
salida = {
    "experimento": "11b_ablacion_ner",
    "descripcion": ("Ablacion del NER: se enmascara el encabezado RECOMENDACIONES en el "
                    "test y se reevalua el modelo YA ENTRENADO, sin reentrenar. Mide si "
                    "el NER aprendio el encabezado o el contenido de la recomendacion."),
    "motivacion": ("La columna Recommendations se reproduce con un regex de una linea al "
                   "99.8%, y el NER da 0.9991. Es el mismo patron que llevo a descartar el "
                   "verificador de BI-RADS, y corresponde aplicar el mismo estandar."),
    "n_test": len(te),
    "config": {"modelo": MODEL_DIR, "max_len": MAX_LEN, "seed": SEED, "device": DEVICE},
    "condicion_A_control":        {k: float(v) for k, v in res_A.items()},
    "condicion_B_sin_encabezado": {k: float(v) for k, v in res_B.items()},
    "referencia_nb11": F1_NB11,
    "caida": float(delta),
    "veredicto": veredicto,
    "contraste_con_birads": {
        "verificador_birads_ablacion": "0.939 -> 0.544 (se derrumba: leia el numero)",
        "ner_ablacion": f"{res_A['f1']:.4f} -> {res_B['f1']:.4f}",
    },
}

Path("resultados").mkdir(exist_ok=True)
with open("resultados/11b_ablacion_ner.json", "w", encoding="utf8") as f:
    json.dump(salida, f, indent=2, ensure_ascii=False)
print("Guardado en resultados/11b_ablacion_ner.json\n")
print(json.dumps({"caida": salida["caida"], "veredicto": veredicto,
                  "contraste": salida["contraste_con_birads"]}, indent=2, ensure_ascii=False))

---
## Conclusiones

> **Completar tras correr el notebook.**

### Las cifras

| Condición | Entrada | F1 de span |
|---|---|---|
| nb 11 | Informe tal cual | 0,9991 |
| **A** | Informe tal cual (reproducción) | `____` |
| **B** | **Sin el encabezado `RECOMENDACIONES:`** | **`____`** |

### El contraste que importa

| Modelo | Ablación | Resultado |
|---|---|---|
| Verificador BI-RADS | Enmascarar el número declarado | 0,939 → **0,544** · se derrumba |
| **NER recomendación** | Enmascarar el encabezado | 0,999 → **`____`** |

### Para la defensa

Si el resultado es H1:

> *"Al auditar el NER encontré que sus etiquetas se derivan de la columna de recomendación del corpus, y que esa columna se reproduce con un regex de una línea al 99,8 %. Es el mismo patrón que me llevó a descartar el verificador de BI-RADS, así que le apliqué la misma prueba: enmascaré el encabezado y reevalué. El verificador de BI-RADS, sin el número, cae de 0,939 a 0,544. El NER, sin el encabezado, se mantiene en ____. La diferencia es que el BI-RADS tiene una sola señal, el número escrito, mientras que la recomendación tiene dos: el encabezado y el contenido. El NER aprendió el contenido, y por eso funciona en los informes chilenos que no traen encabezado."*

Si el resultado es H3:

> *"Le apliqué al NER la misma prueba que al verificador, y no la pasó: sin el encabezado se derrumba. Lo reporto porque el estándar debe ser el mismo para los dos modelos. Su valor descansa en los tres informes chilenos donde funcionó sin encabezado, y eso es poca evidencia. La vía a seguir es reentrenar eliminando el encabezado en parte del entrenamiento, para forzarlo a leer el contenido."*

### Limitaciones

1. Se enmascara **solo el encabezado**, no otras pistas estructurales (la posición al final del informe, el guion inicial). Una ablación más severa las quitaría también.
2. El test sigue siendo el corpus paraguayo. La prueba de generalización real son los informes chilenos.
3. La ablación mide el modelo **tal como está**, entrenado con el encabezado presente. No dice qué pasaría si se entrenara sin él.
